# Advanced Forecasting Techniques for Sales and Purchase Data

This notebook explores a variety of advanced forecasting techniques to handle different types of time series data. We will work with synthetic data that mimics real-world scenarios, including:

- **Sparse Data:** Frequent periods of zero sales, with occasional spikes (e.g., slow-moving or niche products).
- **Volatile Data:** High and unpredictable fluctuations in sales (e.g., products sensitive to market changes or promotions).
- **Stable Data:** Consistent sales with low volatility but potential underlying trends (e.g., staple products).

We will implement and evaluate a suite of models, from classical statistical methods to modern machine learning and deep learning approaches, to determine the best fit for each data type.

## 1. Setup and Dependencies

This notebook requires several libraries. You may need to install them:
```bash
pip install pandas numpy scikit-learn matplotlib seaborn statsmodels prophet lightgbm tensorflow
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Import forecasting models from our script
from forecasting_models import run_forecast

# Settings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

## 2. Synthetic Data Generation

In [ ]:
def generate_base_data(start_date='2022-01-01', periods=365, store_id=101, item_id=12345):
    """Generates a base DataFrame with dates and identifiers."""
    dates = pd.date_range(start=start_date, periods=periods, freq='D')
    df = pd.DataFrame({'Proc_date': dates})
    df['store_id'] = store_id
    df['item'] = item_id
    df['Item_Description'] = 'SYNTHETIC ITEM'
    df['Size'] = '1 EA'
    return df

def generate_sparse_data(periods=365):
    """Generates sparse data with many zeros and occasional spikes."""
    df = generate_base_data(periods=periods)
    is_sale_day = np.random.choice([0, 1], size=periods, p=[0.8, 0.2])
    sales_values = np.random.randint(10, 50, size=periods)
    df['Total_units'] = is_sale_day * sales_values
    df['Total_Retail_] = df['Total_units'] * np.random.uniform(5, 10)
    df['Total_Cost_] = df['Total_Retail_] * np.random.uniform(0.6, 0.8)
    df['Total_lbs'] = 0
    return df.set_index('Proc_date')

def generate_volatile_data(periods=365):
    """Generates volatile data with high, random fluctuations."""
    df = generate_base_data(periods=periods)
    base_sales = 20
    noise = np.random.normal(0, 15, periods)
    df['Total_units'] = (base_sales + noise).astype(int).clip(0)
    df['Total_Retail_] = df['Total_units'] * np.random.uniform(5, 10)
    df['Total_Cost_] = df['Total_Retail_] * np.random.uniform(0.6, 0.8)
    df['Total_lbs'] = 0
    return df.set_index('Proc_date')

def generate_stable_data(periods=365):
    """Generates stable data with a clear trend and low noise."""
    df = generate_base_data(periods=periods)
    time = np.arange(periods)
    trend = 0.1 * time
    seasonality = 5 * np.sin(2 * np.pi * time / 30.5)
    noise = np.random.normal(0, 2, periods)
    df['Total_units'] = (20 + trend + seasonality + noise).astype(int).clip(5)
    df['Total_Retail_] = df['Total_units'] * np.random.uniform(5, 10)
    df['Total_Cost_] = df['Total_Retail_] * np.random.uniform(0.6, 0.8)
    df['Total_lbs'] = 0
    return df.set_index('Proc_date')

## 3. Forecasting Experiments

In [ ]:
def run_experiments(data, models_to_run):
    """A helper function to run a set of models on a given dataset."""
    train, test = train_test_split(data, test_size=0.2, shuffle=False)
    
    results = {}
    plt.figure(figsize=(15, 8))
    plt.plot(train.index, train['Total_units'], label='Train')
    plt.plot(test.index, test['Total_units'], label='Test', color='gray')

    for model_name in models_to_run:
        preds, metrics = run_forecast(model_name, train, test)
        results[model_name] = metrics
        plt.plot(test.index, preds, label=f'{model_name} Forecast')
    
    plt.legend()
    plt.title(f'Forecasts on {data.name} Data')
    plt.show()
    
    results_df = pd.DataFrame(results).T
    print(f"--- Results for {data.name} Data ---")
    print(results_df)
    return results_df

### 3.1 Forecasting on Sparse Data
For sparse data, we'll test models designed for intermittency (Croston, Zero-Inflated) against a standard ML model.

In [ ]:
sparse_df = generate_sparse_data()
sparse_df.name = 'Sparse'
sparse_models = ['Croston', 'Zero-Inflated', 'RandomForest']
sparse_results = run_experiments(sparse_df, sparse_models)

### 3.2 Forecasting on Volatile Data
For volatile data, we'll use robust models that can handle noise well, like LightGBM and Prophet.

In [ ]:
volatile_df = generate_volatile_data()
volatile_df.name = 'Volatile'
volatile_models = ['LightGBM', 'RandomForest', 'Prophet']
volatile_results = run_experiments(volatile_df, volatile_models)

### 3.3 Forecasting on Stable Data
For stable data with trends, classical models like ARIMA and ETS are strong candidates.

In [ ]:
stable_df = generate_stable_data()
stable_df.name = 'Stable'
stable_models = ['ARIMA', 'ETS', 'LightGBM']
stable_results = run_experiments(stable_df, stable_models)

## 4. Applying to Purchase Data (Revenue)
The same functions can be used on purchase data. Let's demonstrate with the `revenue.csv` file, which we'll treat as 'stable' for this example.

In [ ]:
try:
    purchase_df = pd.read_csv('revenue.csv', index_col='Proc_date', parse_dates=True)
    # Resample to daily frequency to fill gaps, then forward-fill data
    purchase_df = purchase_df.asfreq('D').fillna(method='ffill')
    purchase_df.name = 'Purchase'
    
    print("\n--- Running experiments on actual Purchase Data ---")
    purchase_models = ['ARIMA', 'ETS', 'RandomForest']
    purchase_results = run_experiments(purchase_df, purchase_models)
except FileNotFoundError:
    print("revenue.csv not found. Skipping purchase data section.")
except Exception as e:
    print(f"An error occurred while processing purchase data: {e}")